# Project 19 — Tiny VLM (image → encoder → projection → LLM prefix)

```
synthetic RGB 24x24 (shapes/colors) + bar charts + upscaled sklearn digits (OCR) + 3-frame motion clips (video)
  -> tiny CNN encoder -> linear projection -> K visual tokens (prefix)
  -> question word-tokens + visual prefix -> 2-layer Transformer -> 1-token answer head
  -> baselines: full VLM vs text-only (vis=0) vs image-only (txt=0); ablations: K=1/8, noise, frozen encoder
  -> metrics: per-task + joint accuracy, CE; artifacts: results/vlm.csv + ablations.csv + vlm.png
```

Goal: learn the VLM wiring (encoder → projection → prefix fusion → answer head) with numbers,
not slogans: show vision is necessary (text-only ≈ chance), the question matters (image-only drops),
and capacity/robustness tradeoffs (tokens, noise, frozen encoder).

Learning objectives:
1. Draw image → encoder → projection → visual-prefix → LM → answer from memory and name each tensor shape.
2. Explain why prefix fusion conditions generation on vision (keys/values carry image info to every text position).
3. Run QA / OCR / chart / video probes and state which needs vision most (all of them here, by construction).
4. Read the ablation table and argue token-count vs accuracy vs noise-robustness tradeoffs.
  Key lesson: the projection + prefix turn pixels into the LM's language; without them the LM guesses.


In [ ]:
import os # need os to make results/ and read cpu count
import random # need random to seed stdlib RNG
import time # need time to time training + eval
import csv # need csv to write vlm.csv + ablations.csv
import numpy as np # need numpy for synthetic rendering + metrics
import torch # need torch for encoder + transformer + training
import matplotlib # need matplotlib for figures
matplotlib.use('Agg') # force headless PNG backend (no display needed)
import matplotlib.pyplot as plt # need pyplot for accuracy bars + examples
from sklearn.datasets import load_digits # offline OCR proxy: bundled 8x8 digits (no download)
SEED = 19 # fixed seed: same pixels + same init every run
random.seed(SEED) # seed stdlib
np.random.seed(SEED) # seed numpy
torch.manual_seed(SEED) # seed torch
torch.set_num_threads(max(1, os.cpu_count() // 2)) # cap threads: stable timing
os.makedirs('./results', exist_ok=True) # create results/ for CSVs and PNGs
DEV = 'cuda' if torch.cuda.is_available() else 'cpu' # GPU if present else CPU (tiny model: CPU is fine)
print(f"device={DEV} torch={torch.__version__}") # env line for the README


In [ ]:
# ---- vocab: 12 word tokens for questions, 24 answer classes ----
WORDS = ['<pad>','what','shape','color','is','the','which','bar','tallest','digit','direction','motion'] # question word list (index = id)
W2I = {w: i for i, w in enumerate(WORDS)} # word -> id map
V = len(WORDS) # vocab size = 12
Q = 4 # question length: pad every question to 4 tokens
ANS = ['circle','square','triangle','red','green','blue','bar0','bar1','bar2','bar3'] # answers 0..9: shapes, colors, chart bars
ANS += [f'd{i}' for i in range(10)] # answers 10..19: OCR digits d0..d9
ANS += ['left','right','up','down'] # answers 20..23: video motion directions
A2I = {a: i for i, a in enumerate(ANS)} # answer string -> class id
A = len(ANS) # answer classes = 24 (chance joint ~ 1/24 only if uniform; true chance higher per task)
SHAPES = ['circle','square','triangle'] # shape task answer set
COLORS = ['red','green','blue'] # color task answer set
CCOL = {'red': (1,0,0), 'green': (0,1,0), 'blue': (0,0,1)} # color name -> RGB tuple
H = 24 # image height: unify every modality at 3x24x24
# ---- renderer 1: colored shape on black canvas ----
def render_shape(shape, color, rng): # draw one 3x24x24 shape image
    img = np.zeros((3, H, H), dtype=np.float32) # black canvas
    yy, xx = np.mgrid[0:H, 0:H] # pixel coordinate grids
    if shape == 'circle': # disk mask
        m = (yy - 12) ** 2 + (xx - 12) ** 2 <= 49 # radius-7 disk at center
    elif shape == 'square': # box mask
        m = (np.abs(yy - 12) <= 5) & (np.abs(xx - 12) <= 5) # 11x11 centered box
    else: # triangle mask: width grows with row
        m = (yy >= 6) & (yy <= 18) & (np.abs(xx - 12) * 2 <= (yy - 6)) # upright triangle
    r, g, b = CCOL[color] # unpack color channels
    img[0][m], img[1][m], img[2][m] = r, g, b # paint masked pixels
    return img # 3x24x24 float32 in [0,1]
# ---- renderer 2: 4-bar chart with unique max ----
def render_chart(rng): # draw 4 white bars, return (image, argmax label)
    img = np.zeros((3, H, H), dtype=np.float32) # black canvas
    hs = rng.randint(1, 5, size=4) # 4 bar heights in {1..4} (x4 px each)
    while len(set(hs.tolist())) < 4 or list(hs).count(max(hs)) > 1: # force a unique max (unambiguous label)
        hs = rng.randint(1, 5, size=4) # resample ties away
    for i, h in enumerate(hs): # paint each bar
        x0 = 2 + i * 6 # bar left edge (width 4, gap 2)
        img[:, H - int(h) * 4:, x0:x0 + 4] = 1.0 # white bar from bottom up
    return img, int(np.argmax(hs)) # image + tallest-bar index 0..3
# ---- renderer 3: OCR digits from sklearn (offline, bundled) ----
_digits = load_digits() # 1797 8x8 gray digits, ships with sklearn
_dX = (_digits.images / 16.0).astype(np.float32) # normalize 0..16 -> 0..1
_dY = _digits.target.astype(np.int64) # digit labels 0..9
def render_digit(idx): # upscale 8x8 -> 24x24 RGB by 3x kron repeat
    g = np.kron(_dX[idx], np.ones((3, 3), dtype=np.float32)) # 24x24 gray
    return np.stack([g, g, g], axis=0), int(_dY[idx]) # repeat to 3 channels + label
# ---- renderer 4: video = 3-frame dot motion stacked as 3 channels ----
def render_video(direction, rng): # dot steps 4px/frame along direction
    fr = np.zeros((3, H, H), dtype=np.float32) # 3 frames = 3 channels
    d = {'left': (0,-4), 'right': (0,4), 'up': (-4,0), 'down': (4,0)}[direction] # per-frame delta
    yy, xx = np.mgrid[0:H, 0:H] # coordinate grids for the dot mask
    for t in range(3): # one dot position per frame
        cy, cx = 12 + d[0] * t, 12 + d[1] * t # center drifts with t
        fr[t] = (((yy - cy) ** 2 + (xx - cx) ** 2) <= 4).astype(np.float32) # radius-2 dot in channel t
    return fr # motion lives in cross-channel displacement
# ---- question encoder: words -> Q=4 ids ----
def qids(words): # map word list to padded id vector
    ids = [W2I[w] for w in words] # look up each word
    return np.array(ids + [0] * (Q - len(ids)), dtype=np.int64) # right-pad with <pad> to Q=4
QSHAPE = qids(['what','shape','is','the']) # shape question
QCOLOR = qids(['what','color','is','the']) # color question
QCHART = qids(['which','bar','tallest']) # chart question (3 words + pad)
QDIGIT = qids(['what','digit','is','the']) # OCR question
QVIDEO = qids(['which','direction','motion']) # video question
# ---- dataset: multitask mix with task ids 0..4 ----
def build_data(n_shape, n_chart, n_digit, n_video, seed): # seeded render for a split
    rng = np.random.RandomState(seed) # split-local RNG (train vs val differ)
    imgs, qs, ys, ts = [], [], [], [] # collectors
    for _ in range(n_shape): # shape+color QA share images, differ by question
        s = SHAPES[rng.randint(3)] # random shape
        c = list(CCOL)[rng.randint(3)] # random color
        im = render_shape(s, c, rng) # render once
        if rng.rand() < 0.5: # ask shape half the time
            imgs.append(im); qs.append(QSHAPE); ys.append(A2I[s]); ts.append(0) # task 0 = shape
        else: # ask color otherwise
            imgs.append(im); qs.append(QCOLOR); ys.append(A2I[c]); ts.append(1) # task 1 = color
    for _ in range(n_chart): # chart samples
        im, b = render_chart(rng) # unique-max chart
        imgs.append(im); qs.append(QCHART); ys.append(A2I[f'bar{b}']); ts.append(2) # task 2 = chart
    perm = np.random.RandomState(seed + 1).permutation(len(_dX)) # shuffled digit order per split
    for k in range(n_digit): # OCR samples
        im, d = render_digit(int(perm[k % len(_dX)])) # cycle through digits
        imgs.append(im); qs.append(QDIGIT); ys.append(A2I[f'd{d}']); ts.append(3) # task 3 = ocr
    for _ in range(n_video): # video samples
        di = ['left','right','up','down'][rng.randint(4)] # random direction
        imgs.append(render_video(di, rng)); qs.append(QVIDEO); ys.append(A2I[di]); ts.append(4) # task 4 = video
    X = torch.from_numpy(np.stack(imgs)) # N x 3 x 24 x 24
    Qid = torch.from_numpy(np.stack(qs)) # N x 4
    Y = torch.tensor(ys, dtype=torch.long) # N answers
    T = torch.tensor(ts, dtype=torch.long) # N task ids
    return X, Qid, Y, T # full split tensors
Xtr, Qtr, Ytr, Ttr = build_data(900, 400, 1400, 400, 100) # train split (3100 renders)
Xva, Qva, Yva, Tva = build_data(180, 100, 397, 100, 200) # val split (777 renders, fresh RNG)
print(f"train={len(Xtr)} val={len(Xva)} img={tuple(Xtr.shape[1:])} vocab={V} answers={A}") # shape card
assert Xtr.shape[1:] == (3, 24, 24) # every modality unified at 3x24x24
assert set(Ttr.tolist()) == {0, 1, 2, 3, 4} # all 5 tasks present in train


In [ ]:
# ---- model: CNN encoder -> projection -> visual prefix + question -> tiny transformer -> answer ----
class TinyVLM(torch.nn.Module): # image + question -> 1-token answer logits
    def __init__(self, n_vis=4, d=64): # n_vis visual tokens, d model width
        super().__init__() # init nn.Module
        self.n_vis, self.d = n_vis, d # stash prefix length + width
        self.conv1 = torch.nn.Conv2d(3, 16, 3, stride=2, padding=1) # 24->12 spatial downsample
        self.conv2 = torch.nn.Conv2d(16, 32, 3, stride=2, padding=1) # 12->6 deeper features
        self.fc1 = torch.nn.Linear(32 * 6 * 6, 256) # flatten 1152 -> 256
        self.proj = torch.nn.Linear(256, n_vis * d) # projection: vision -> K tokens x d (the VLM bridge)
        self.wemb = torch.nn.Embedding(V, d) # question word embeddings
        self.pos = torch.nn.Embedding(n_vis + Q, d) # learned positions over [vis | question]
        lyr = torch.nn.TransformerEncoderLayer(d, 4, 128, dropout=0.0, batch_first=True) # 1 pre-norm block
        self.tr = torch.nn.TransformerEncoder(lyr, 2) # 2-layer tiny LM (prefix fusion happens here)
        self.head = torch.nn.Linear(d, A) # answer head: pooled state -> 24 classes
    def encode_vis(self, img): # pixels -> (B, n_vis, d) visual tokens
        h = torch.relu(self.conv1(img)) # B x 16 x 12 x 12
        h = torch.relu(self.conv2(h)) # B x 32 x 6 x 6
        h = torch.relu(self.fc1(h.flatten(1))) # B x 256
        return self.proj(h).view(-1, self.n_vis, self.d) # B x K x d visual prefix
    def forward(self, img, q, zero_vis=False, zero_txt=False): # flags implement the two ablated baselines
        B = img.size(0) # batch size
        vis = self.encode_vis(img) # B x K x d
        if zero_vis: # text-only baseline: kill vision, keep shapes
            vis = torch.zeros_like(vis) # LM must guess from question alone
        txt = self.wemb(q) # B x 4 x d question embeddings
        if zero_txt: # image-only baseline: kill the question
            txt = torch.zeros_like(txt) # head sees pixels but not the task
        seq = torch.cat([vis, txt], dim=1) # prefix fusion: [vis K | txt 4] along time
        pos = torch.arange(self.n_vis + Q, device=img.device) # position ids 0..K+3
        seq = seq + self.pos(pos).unsqueeze(0) # add learned positions
        h = self.tr(seq) # contextualize: every text pos attends to every visual token
        pooled = h[:, self.n_vis:, :].mean(1) # mean-pool the 4 question positions
        return self.head(pooled) # B x 24 answer logits
def count_params(m): # total trainable params
    return sum(p.numel() for p in m.parameters() if p.requires_grad) # scalar count
m0 = TinyVLM(4, 64) # probe instance for shape tests
with torch.no_grad(): # no graph for unit tests
    pv = m0.encode_vis(Xva[:8]) # 8 val images -> visual tokens
    assert tuple(pv.shape) == (8, 4, 64) # projection contract: (B, K, d)
    l1 = m0(Xva[:8], Qva[:8]) # full forward
    assert tuple(l1.shape) == (8, A) # head contract: (B, 24)
    l2 = m0(Xva[:8], Qva[:8]) # second forward, same weights
    assert torch.equal(l1, l2) # eval-mode determinism (dropout=0)
    assert torch.isfinite(l1).all() # no NaN/Inf from renderers
print(f"params={count_params(m0)} prefix=(B,4,64) head->({A})") # param card for README


In [ ]:
# ---- train: multitask CE; eval: per-task + joint accuracy ----
def run_epoch(model, opt, X, Qd, Y, train): # one pass; train=True steps Adam
    model.train(train) # set mode (dropout=0 anyway; keeps semantics honest)
    tot, loss_sum, correct = 0, 0.0, 0 # accumulators
    idx = torch.randperm(len(X)) if train else torch.arange(len(X)) # shuffle iff training
    for s in range(0, len(X), 64): # minibatches of 64
        b = idx[s:s + 64] # batch indices
        xb, qb, yb = X[b].to(DEV), Qd[b].to(DEV), Y[b].to(DEV) # move batch to device
        logits = model(xb, qb) # B x 24
        loss = torch.nn.functional.cross_entropy(logits, yb) # multitask CE on the answer token
        if train: # backward + step only in train mode
            opt.zero_grad() # clear stale grads
            loss.backward() # backprop through head + transformer + projection + CNN
            opt.step() # Adam update
        loss_sum += loss.item() * len(b) # weight by batch size
        correct += (logits.argmax(1) == yb).sum().item() # top-1 hits
        tot += len(b) # count samples
    return loss_sum / tot, correct / tot # mean CE + accuracy
def eval_tasks(model, X, Qd, Y, T): # per-task accuracy + joint
    model.eval() # eval mode
    out = {} # task -> acc
    with torch.no_grad(): # no graph
        for t in range(5): # tasks 0..4
            m = (T == t) # mask for this task
            lo = model(X[m].to(DEV), Qd[m].to(DEV)) # logits on CPU/GPU
            out[t] = (lo.argmax(1).cpu() == Y[m]).float().mean().item() # accuracy
        lo = model(X.to(DEV), Qd.to(DEV)) # all val
        out['joint'] = (lo.argmax(1).cpu() == Y).float().mean().item() # joint accuracy
        out['ce'] = torch.nn.functional.cross_entropy(lo, Y.to(DEV)).item() # val CE
    return out # dict of 5 tasks + joint + ce
TNAMES = ['shape', 'color', 'chart', 'ocr', 'video'] # task index -> name
def train_model(n_vis=4, epochs=8, frozen_enc=False, seed=0): # full train helper (baselines + ablations reuse it)
    torch.manual_seed(seed) # fresh but deterministic init per config
    m = TinyVLM(n_vis).to(DEV) # build at this prefix length
    if frozen_enc: # frozen-encoder ablation: random CNN stays random
        for n, p in m.named_parameters(): # walk params
            if n.startswith('conv') or n.startswith('fc1'): # encoder params only
                p.requires_grad = False # freeze them (head + transformer still learn)
    opt = torch.optim.Adam([p for p in m.parameters() if p.requires_grad], lr=3e-3) # Adam over unfrozen
    for e in range(epochs): # epoch loop
        run_epoch(m, opt, Xtr, Qtr, Ytr, True) # one training pass (loss printed only at end)
    return m # trained model
t0 = time.time() # wall clock for the training block
full = train_model(4, 8, False, 0) # VLM: K=4, 8 epochs
txt_only = train_model(4, 8, False, 1) # same capacity, evaluated with vis=0 (separate seed = honest baseline)
img_only = train_model(4, 8, False, 2) # same capacity, evaluated with txt=0
r_full = eval_tasks(full, Xva, Qva, Yva, Tva) # VLM scores
with torch.no_grad(): # baseline evals need no graph
    txt_only.eval() # eval mode
    lo = txt_only(Xva.to(DEV), Qva.to(DEV), zero_vis=True) # kill vision at eval
    r_txt = {'joint': (lo.argmax(1).cpu() == Yva).float().mean().item()} # text-only joint
    img_only.eval() # eval mode
    lo = img_only(Xva.to(DEV), Qva.to(DEV), zero_txt=True) # kill question at eval
    r_img = {'joint': (lo.argmax(1).cpu() == Yva).float().mean().item()} # image-only joint
    r_img01 = [] # shape+color accs without the question (question hidden => ~0.5 ceiling)
    for t in (0, 1): # ambiguous tasks share images, differ only by question
        m = (Tva == t) # mask for this task
        r_img01.append((lo.argmax(1).cpu()[m] == Yva[m]).float().mean().item()) # per-task acc
print(f"VLM joint={r_full['joint']:.3f} ce={r_full['ce']:.3f} " + ' '.join(f"{TNAMES[t]}={r_full[t]:.3f}" for t in range(5))) # VLM card
print(f"text-only={r_txt['joint']:.3f} image-only={r_img['joint']:.3f} img-shape={r_img01[0]:.3f} img-color={r_img01[1]:.3f} ({time.time()-t0:.1f}s train)") # baseline card
assert r_full['joint'] > r_txt['joint'] + 0.30 # vision must carry >30 pts (guards a broken fusion)
assert r_full['joint'] > r_img['joint'] + 0.10 # the question must matter (joint margin on top of per-task proof below)
assert r_full[0] >= 0.95 and r_img01[0] <= 0.75 # shape needs the question: VLM ~1.0, blind ~0.5
assert r_full[1] >= 0.95 and r_img01[1] <= 0.75 # color needs the question: same argument
assert r_full[2] >= 0.90 and r_full[4] >= 0.85 # chart + video are easy: near-ceiling expected


In [ ]:
# ---- what-if: prefix length K=1/8, pixel noise, frozen encoder; save CSVs + figure ----
m1 = train_model(1, 5, False, 10) # K=1 bottleneck, 5 epochs
r1 = eval_tasks(m1, Xva, Qva, Yva, Tva) # its val scores
m8 = train_model(8, 5, False, 11) # K=8 roomy prefix, 5 epochs
r8 = eval_tasks(m8, Xva, Qva, Yva, Tva) # its val scores
mfr = train_model(4, 5, True, 12) # frozen random encoder, train head+transformer only
rfr = eval_tasks(mfr, Xva, Qva, Yva, Tva) # its val scores
full.eval() # eval mode for the noise probe
with torch.no_grad(): # no graph
    Xn = (Xva + 0.25 * torch.randn_like(Xva)).clamp(0, 1) # noisy val: sigma=0.25 Gaussian
    lon = full(Xn.to(DEV), Qva.to(DEV)) # forward on corrupted pixels
    r_noise = (lon.argmax(1).cpu() == Yva).float().mean().item() # noisy joint accuracy
print(f"K=1 joint={r1['joint']:.3f} K=8 joint={r8['joint']:.3f} frozen={rfr['joint']:.3f} noisy={r_noise:.3f} clean={r_full['joint']:.3f}") # ablation card
assert r1['joint'] > 0.50 and r8['joint'] > 0.50 # both capacities far above chance (~0.1): training worked
assert rfr['joint'] > 0.30 # frozen CNN still learns a linear-ish readout (weak but real)
assert r_noise <= r_full['joint'] + 1e-9 # noise must not help (sanity on the probe)
rows = [{'model': 'vlm-K4', 'task': TNAMES[t], 'acc': round(r_full[t], 4), 'n': int((Tva == t).sum())} for t in range(5)] # per-task rows
rows += [{'model': 'vlm-K4', 'task': 'joint', 'acc': round(r_full['joint'], 4), 'n': len(Yva)}] # joint row
rows += [{'model': 'text-only', 'task': 'joint', 'acc': round(r_txt['joint'], 4), 'n': len(Yva)}] # baseline row
rows += [{'model': 'image-only', 'task': 'joint', 'acc': round(r_img['joint'], 4), 'n': len(Yva)}] # baseline row
with open('./results/vlm.csv', 'w', newline='') as f: # save main table
    w = csv.DictWriter(f, fieldnames=['model', 'task', 'acc', 'n']) # header from keys
    w.writeheader() # header once
    w.writerows(rows) # per-task + baselines
print('saved ./results/vlm.csv') # confirm artifact
ab = [{'config': 'K=1', 'joint': round(r1['joint'], 4)}, {'config': 'K=4-full', 'joint': round(r_full['joint'], 4)}] # token sweep rows
ab += [{'config': 'K=8', 'joint': round(r8['joint'], 4)}, {'config': 'frozen-enc', 'joint': round(rfr['joint'], 4)}] # capacity + frozen rows
ab += [{'config': 'noise-s0.25', 'joint': round(r_noise, 4)}] # robustness row
with open('./results/ablations.csv', 'w', newline='') as f: # save ablation table
    w = csv.DictWriter(f, fieldnames=['config', 'joint']) # header from keys
    w.writeheader() # header once
    w.writerows(ab) # one row per config
print('saved ./results/ablations.csv') # confirm artifact
fig, ax = plt.subplots(1, 3, figsize=(15, 4)) # three panels: tasks, ablations, examples
xs = np.arange(5) # task positions
ax[0].bar(xs - 0.2, [r_full[t] for t in range(5)], 0.2, label='VLM') # full model per task
ax[0].bar(xs, [r1[t] for t in range(5)], 0.2, label='K=1') # bottleneck per task
ax[0].bar(xs + 0.2, [r8[t] for t in range(5)], 0.2, label='K=8') # roomy per task
ax[0].set_xticks(xs, TNAMES, rotation=15) # task labels
ax[0].set_ylim(0, 1.05) # accuracy range
ax[0].set_ylabel('accuracy') # y label
ax[0].set_title('P19: per-task accuracy') # panel title
ax[0].legend() # show series
ax[1].bar([r['config'] for r in ab], [r['joint'] for r in ab]) # joint per config
ax[1].tick_params(axis='x', rotation=20) # tilt labels
ax[1].set_ylim(0, 1.05) # accuracy range
ax[1].set_ylabel('joint accuracy') # y label
ax[1].set_title('P19: ablations (tokens/frozen/noise)') # panel title
ex = [Xva[(Tva == 0)][0].permute(1, 2, 0).numpy(), Xva[(Tva == 2)][0].permute(1, 2, 0).numpy()] # shape + chart examples
ex.append(Xva[(Tva == 3)][0].permute(1, 2, 0).numpy()) # digit example
ex.append(Xva[(Tva == 4)][0][1].numpy()) # video middle frame (channel 1)
for i, im in enumerate(ex): # four example thumbnails
    ax[2].inset_axes([0.02 + i * 0.245, 0.15, 0.22, 0.7]).imshow(im.squeeze(), vmin=0, vmax=1) # tiny axes per example
    ax[2].inset_axes([0.02 + i * 0.245, 0.15, 0.22, 0.7]).axis('off') # no ticks on thumbnails
ax[2].set_title('P19: shape | chart | digit | video-frame') # panel title
ax[2].axis('off') # no axes on the gallery panel
fig.tight_layout() # avoid overlap
fig.savefig('./results/vlm.png', dpi=100) # save figure
print('saved ./results/vlm.png') # confirm artifact


Cleanup / next steps:
- Artifacts: `results/vlm.csv` (per-task + baselines), `results/ablations.csv` (tokens/frozen/noise),
  `results/vlm.png` (accuracy bars + example pixels).
- Rerun on DGX Spark with identical seeds; numbers should match to noise (CPU-quantized ops aside).
- Next: P20 builds RAG from scratch (chunk → embed → index → retrieve → prompt) and splits
  retrieval quality from generation quality — the text-only-LM baseline here becomes the closed-book arm there.
